# Hands-on Projects (Image Generation)

**Module:** 17 — Image Generation

Four projects: prompt library CLI, inpaint tool, LoRA eval sheet, and cost dashboard.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Implement scaffolded projects with acceptance tests
- Practice prompt ops, editing, evaluation, and cost control
- Document failure modes and SLOs for each deliverable


## Project map

```mermaid
flowchart LR
  P1[Prompt Library CLI] --> P2[Inpaint Tool]
  P2 --> P3[LoRA Eval Sheet]
  P3 --> P4[Cost Dashboard]
```


## Project 1 — Prompt Library CLI

**Brief:** Named prompt templates with slots, brand version, negatives.

| ID | Acceptance |
|----|------------|
| A1 | add/get/list on JSON store |
| A2 | Slot render fills `{subject}` |
| A3 | Brand version on every render |


In [ ]:
# Project 1 starter
from pathlib import Path
import json

LIB = Path("prompt_lib.json")

def load():
    return json.loads(LIB.read_text(encoding="utf-8")) if LIB.exists() else {}

def save(data):
    LIB.write_text(json.dumps(data, indent=2), encoding="utf-8")

def add(name, template, negative, brand_version):
    data = load(); data[name] = {"template": template, "negative": negative, "brand_version": brand_version}; save(data)

def render(name, **slots):
    item = load()[name]
    return {"prompt": item["template"].format(**slots), "negative": item["negative"], "brand_version": item["brand_version"]}

add("product_hero", "{subject} on marble, softbox, {brand_style}", "watermark, blurry", "2026.04")
print(render("product_hero", subject="teal mug", brand_style="flat coral accents"))


### Try it yourself — Project 1

1. Add diff between template versions.
2. Validate required slots.
3. Export golden suite CSV.


## Project 2 — Inpaint Tool

| ID | Acceptance |
|----|------------|
| B1 | Reject strength>0.8 for identity-critical |
| B2 | Feather recommendation from mask coverage |
| B3 | Emit worker job JSON |


In [ ]:
# Project 2 starter
def plan_inpaint(prompt, strength, mask_coverage, identity_critical):
    if identity_critical and strength > 0.8:
        return {"status": "reject", "reason": "strength_too_high"}
    feather = 16 if mask_coverage < 0.1 else 8
    return {"status": "ok",
            "job": {"mode": "inpaint", "prompt": prompt, "strength": strength, "feather_px": feather},
            "qa_thresholds": {"identity": 0.85 if identity_critical else 0.7}}

print(plan_inpaint("replace background", 0.9, 0.4, True))
print(plan_inpaint("replace mug", 0.55, 0.05, True))


### Try it yourself — Project 2

1. Add seam score mock.
2. Support outpaint expand math.
3. Wire provider interface from nb06.


## Project 3 — LoRA Evaluation Sheet

Score LoRAs on identity, style, promptability, artifacts.


In [ ]:
# Project 3 starter
def lora_score(row):
    w = {"identity": 0.35, "style": 0.25, "promptability": 0.25, "low_artifacts": 0.15}
    return sum(row[k] * w[k] for k in w)

rows = {
    "lora_a": {"identity": 0.9, "style": 0.7, "promptability": 0.8, "low_artifacts": 0.85},
    "lora_b": {"identity": 0.7, "style": 0.95, "promptability": 0.6, "low_artifacts": 0.7},
}
print(sorted(((k, round(lora_score(v), 3)) for k, v in rows.items()), key=lambda x: -x[1]))


## Project 4 — Cost Dashboard

Estimate monthly spend; alert on budget.


In [ ]:
# Project 4 starter
def month_cost(qps, seconds_per_img, price, hours=24*30, retry=0.1):
    images = qps * 3600 * hours
    return {"images": int(images), "usd": round(images * (1 + retry) * price, 2)}

budget = 5000
est = month_cost(0.3, 5, 0.04)
print(est, "OVER" if est["usd"] > budget else "OK")


## Acceptance Checklist
- [ ] Inline docs in notebook (no separate README files)
- [ ] ≥5 asserts/smoke tests
- [ ] Failure mode list per project
- [ ] Params logged in job JSON
- [ ] Safety note for uploads


In [ ]:
# Shared smoke tests
assert render("product_hero", subject="x", brand_style="y")["brand_version"] == "2026.04"
assert plan_inpaint("bg", 0.9, 0.2, True)["status"] == "reject"
assert lora_score(rows["lora_a"]) > lora_score(rows["lora_b"])
assert month_cost(0.01, 5, 0.04)["usd"] > 0
print("smoke tests passed")


### Try it yourself — Ship

1. Persist Project 1 history with timestamps.
2. Add Prometheus-style metric names for Project 4.

**Stretch:** Bake off two real APIs with YOUR_* keys on 10 golden prompts.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `golden suite` | Fixed prompts/images for regression |
| `promptability` | Personalized model still follows prompts |
| `unit economics` | Cost per accepted image at target quality |


### Workshop — Parameter journal — Image Projects

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Image Projects
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Image Projects

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Image Projects
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Image Projects

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Image Projects
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Image Projects

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Image Projects
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Image Projects

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Image Projects
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Image Projects

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Image Projects
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


## Key Takeaways

- Projects should exercise prompts, edits, eval, and cost
- Acceptance tests beat screenshot galleries
- Log provenance on every generated asset
